# Relative Active Graph — C4 Training Notebook

Train the **MTLG lexicon** and **TRD bootstrapper** on a streaming subset of the English [C4 corpus](https://huggingface.co/datasets/allenai/c4) from HuggingFace, then demonstrate inference on new sentences.

### Pipeline
1. **Stream** C4 (English, ~500 sentences) via HuggingFace `datasets` (streaming mode — no full download needed)
2. **Parse** with Stanza Universal Dependencies
3. **Convert** UD trees → MTLG modal graphs (◇ diamond / □ box / ◊ lozenge modes)
4. **Induce** probabilistic MTLG lexicon (Kwiatkowski-style, multimodal extension)
5. **Bootstrap** TRD clusters (k-means over modal type profiles)
6. **Evaluate** parse accuracy + TRD coverage on held-out sentences
7. **Inference demo** — analyze new sentences with the trained model

> **Runtime tip:** GPU is not required; the demo completes in ~5 min on a standard Colab CPU.


## 1 · Install Dependencies

In [ ]:
%%capture
!pip install datasets stanza numpy scipy networkx sentencepiece matplotlib tqdm
print("Dependencies installed.")


## 2 · Write Source Modules

The RAG induction pipeline lives in `lcs/induction/`. We write each module to the Colab working directory so they can be imported normally.


In [ ]:
%%writefile mc4_stream.py
"""Stream sentences from the mC4 multilingual corpus (allenai/c4, mC4 variant).

Yields dicts: {"text": str, "language": str, "url": str}.
Supports 108 languages. Streaming mode avoids downloading the full dataset.
"""
from __future__ import annotations

import logging
from typing import Iterator

logger = logging.getLogger(__name__)

# Languages in the mC4 corpus (representative subset; full list has 108).
MC4_LANGUAGES = [
    "af", "am", "ar", "az", "be", "bg", "bn", "ca", "cs", "cy",
    "da", "de", "el", "en", "eo", "es", "et", "eu", "fa", "fi",
    "fr", "fy", "ga", "gl", "gu", "ha", "hi", "hr", "ht", "hu",
    "hy", "id", "ig", "is", "it", "iw", "ja", "ka", "kk", "km",
    "kn", "ko", "ku", "ky", "la", "lb", "lo", "lt", "lv", "mg",
    "mi", "mk", "ml", "mn", "mr", "ms", "mt", "my", "ne", "nl",
    "no", "ny", "pa", "pl", "ps", "pt", "ro", "ru", "sd", "si",
    "sk", "sl", "sm", "sn", "so", "sq", "sr", "st", "su", "sv",
    "sw", "ta", "te", "tg", "th", "tk", "tl", "tr", "tt", "ug",
    "uk", "ur", "uz", "vi", "xh", "yi", "yo", "zh", "zu",
]

# Languages requiring FST morphological decomposition before type assignment.
AGGLUTINATIVE_LANGUAGES = {"fi", "tr", "hu", "ka", "sw", "tl", "az", "kk", "ky", "uz"}
POLYSYNTHETIC_LANGUAGES  = {"my"}  # Burmese approximation; full list includes Nahuatl etc.
FST_REQUIRED_LANGUAGES   = AGGLUTINATIVE_LANGUAGES | POLYSYNTHETIC_LANGUAGES


def stream_mc4(
    language:    str,
    max_samples: int = 10_000,
    split:       str = "train",
) -> Iterator[dict]:
    """Stream sentences from mC4 for a given language.

    Args:
        language:    ISO 639-1 code (must be in MC4_LANGUAGES).
        max_samples: Maximum sentences to yield (streaming).
        split:       Dataset split ("train" or "validation").

    Yields:
        {"text": str, "language": str, "url": str}
    """
    if language not in MC4_LANGUAGES:
        raise ValueError(f"Language '{language}' not in mC4. Supported: {MC4_LANGUAGES}")

    try:
        from datasets import load_dataset
    except ImportError:
        raise ImportError("Install 'datasets' package: pip install datasets")

    logger.info("Streaming mC4 for language=%s, split=%s, max=%d", language, split, max_samples)

    dataset = load_dataset(
        "allenai/c4",
        name=language,
        split=split,
        streaming=True,
        trust_remote_code=True,
    )

    count = 0
    for example in dataset:
        if count >= max_samples:
            break
        text = example.get("text", "").strip()
        if not text:
            continue
        # Split into sentences (simple heuristic; Stanza handles proper segmentation).
        for sent in _split_sentences(text):
            if count >= max_samples:
                break
            yield {"text": sent, "language": language, "url": example.get("url", "")}
            count += 1

    logger.info("Streamed %d sentences for language=%s", count, language)


def _split_sentences(text: str, max_length: int = 512) -> list[str]:
    """Basic sentence splitting: split on '. ', '! ', '? ' boundaries."""
    import re
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s[:max_length] for s in sents if len(s.split()) >= 3]


def requires_fst(language: str) -> bool:
    """Does this language require FST decomposition before type assignment?"""
    return language in FST_REQUIRED_LANGUAGES


def stream_multilingual(
    languages:   list[str] | None = None,
    max_per_lang: int = 1_000,
) -> Iterator[dict]:
    """Stream sentences interleaved across multiple languages."""
    langs = languages or MC4_LANGUAGES[:10]  # default: first 10 for dev
    for lang in langs:
        try:
            yield from stream_mc4(lang, max_samples=max_per_lang)
        except Exception as exc:
            logger.warning("Skipping language %s: %s", lang, exc)


if __name__ == "__main__":
    import sys
    import json
    lang = sys.argv[1] if len(sys.argv) > 1 else "en"
    for item in stream_mc4(lang, max_samples=5):
        print(json.dumps(item, ensure_ascii=False))


In [ ]:
%%writefile morphological_fst.py
"""FST-based morphological decomposition for agglutinative/polysynthetic languages.

Required before MTLG type assignment for languages in FST_REQUIRED_LANGUAGES.
A single word in Turkish/Finnish can encode a full clause; FST decomposes it
into morpheme sequences that MTLG can then assign types to.

Uses sentencepiece for analytic/fusional languages (fallback).
Uses morphological rules (simplified FST) for agglutinative languages.
"""
from __future__ import annotations

import logging
import re
from dataclasses import dataclass
from typing import Optional

from mc4_stream import FST_REQUIRED_LANGUAGES

logger = logging.getLogger(__name__)


@dataclass
class MorphemeDecomposition:
    """Result of FST decomposition: the word split into typed morphemes."""
    original:   str
    language:   str
    morphemes:  list[str]         # individual morphemes in order
    tags:       list[str]         # morphological tags (STEM, TENSE, PERSON, etc.)
    is_clause_level: bool         # True if this word encodes a full clause


class SimplifiedFst:
    """A simplified Finite-State Transducer for morphological decomposition.

    In production, this would be backed by a full FST library (e.g., rustfst via
    Python bindings, or hfst). Here we implement a rule-based approximation
    sufficient for the CSRRE LCS pipeline.
    """

    # Turkish morpheme boundary patterns (simplified).
    TURKISH_SUFFIXES = [
        (r"(yor)$",    "PROG"),  # progressive
        (r"(di)$",     "PAST"),  # past tense
        (r"(mek)$",    "INF"),   # infinitive
        (r"(ler|lar)$", "PL"),   # plural
        (r"(im|ım|üm|um)$", "1SG"),  # 1st sg
        (r"(sin|sın|sün|sun)$", "2SG"),
        (r"(iz|ız|üz|uz)$", "1PL"),
        (r"(da|de|ta|te)$", "LOC"),
        (r"(dan|den|tan|ten)$", "ABL"),
        (r"(a|e|ya|ye)$", "DAT"),
        (r"(ı|i|u|ü|yı|yi|yu|yü)$", "ACC"),
        (r"(ın|in|un|ün|nın|nin|nun|nün)$", "GEN"),
    ]

    FINNISH_SUFFIXES = [
        (r"(ssa|ssä)$", "INESS"),   # inessive
        (r"(sta|stä)$", "ELAT"),    # elative
        (r"(lle)$",     "ALLAT"),   # allative
        (r"(lla|llä)$", "ADESS"),   # adessive
        (r"(lta|ltä)$", "ABLAT"),   # ablative
        (r"(n)$",       "GEN"),     # genitive
        (r"(t)$",       "PL_NOM"),  # plural nominative
        (r"(ksi)$",     "TRANSL"),  # translative
        (r"(tta|ttä)$", "ABESS"),   # abessive
    ]

    def __init__(self, language: str):
        self.language = language
        self.suffix_rules = self._load_suffix_rules(language)

    def _load_suffix_rules(self, language: str) -> list[tuple[str, str]]:
        if language == "tr":
            return self.TURKISH_SUFFIXES
        elif language == "fi":
            return self.FINNISH_SUFFIXES
        else:
            return []  # other languages: BPE fallback

    def decompose(self, word: str) -> MorphemeDecomposition:
        """Decompose a word into morphemes by iteratively stripping suffixes."""
        remaining = word.lower()
        morphemes = []
        tags = []
        is_clause = False

        for pattern, tag in self.suffix_rules:
            m = re.search(pattern, remaining)
            if m:
                suffix   = m.group(1)
                stem     = remaining[:m.start()]
                remaining = stem
                morphemes.insert(0, suffix)
                tags.insert(0, tag)
                if tag in ("PROG", "PAST", "INF"):
                    is_clause = True

        morphemes.insert(0, remaining)  # stem is first
        tags.insert(0, "STEM")

        return MorphemeDecomposition(
            original=word,
            language=self.language,
            morphemes=morphemes,
            tags=tags,
            is_clause_level=is_clause,
        )

    def decompose_sentence(self, sentence: str) -> list[MorphemeDecomposition]:
        return [self.decompose(w) for w in sentence.split()]


def get_decomposer(language: str) -> Optional["SimplifiedFst"]:
    """Get a decomposer for `language`, or None if not needed (use BPE instead)."""
    if language in FST_REQUIRED_LANGUAGES:
        return SimplifiedFst(language)
    return None


def preprocess_for_type_assignment(sentence: str, language: str) -> list[str]:
    """Convert a sentence to a token sequence suitable for MTLG type assignment.

    For agglutinative/polysynthetic: FST decompose → morpheme sequence.
    For analytic/fusional: sentencepiece BPE tokenization.
    """
    if language in FST_REQUIRED_LANGUAGES:
        fst = SimplifiedFst(language)
        decomps = fst.decompose_sentence(sentence)
        # Flatten to morpheme sequence: stem + each morpheme tag as a pseudo-token.
        tokens = []
        for d in decomps:
            tokens.extend([f"{m}[{t}]" for m, t in zip(d.morphemes, d.tags)])
        return tokens
    else:
        # Analytic/fusional: simple whitespace tokenization (BPE would require sentencepiece model).
        return sentence.split()


if __name__ == "__main__":
    import json
    examples = [
        ("tr", "gidiyorum"),       # Turkish: "I am going"
        ("fi", "talossani"),       # Finnish: "in my house"
        ("en", "the cat sat"),     # English: no FST needed
    ]
    for lang, word in examples:
        decomposer = get_decomposer(lang)
        if decomposer:
            d = decomposer.decompose(word)
            print(json.dumps({"lang": lang, "word": word,
                              "morphemes": d.morphemes, "tags": d.tags,
                              "is_clause": d.is_clause_level}))
        else:
            tokens = preprocess_for_type_assignment(word, lang)
            print(json.dumps({"lang": lang, "word": word, "tokens": tokens, "fst": False}))


In [ ]:
%%writefile ud_parser.py
"""Universal Dependencies (UD) parser wrapper using Stanza.

Produces UD dependency trees from sentences in any of the 108 mC4 languages.
Language-agnostic annotation scheme: the same UD relation set applies to all languages.
"""
from __future__ import annotations

import logging
from dataclasses import dataclass, field
from typing import Optional

logger = logging.getLogger(__name__)

# UD dependency relation types used in MTLG edge assignment.
UD_CORE_DEPS    = {"nsubj", "obj", "iobj", "csubj", "ccomp", "xcomp"}
UD_NONCORE_DEPS = {"obl", "vocative", "expl", "dislocated", "advcl", "advmod",
                   "discourse", "aux", "cop", "mark"}
UD_MODIFIER_DEPS = {"nmod", "appos", "nummod", "acl", "amod", "det", "clf"}
UD_SPECIAL_DEPS  = {"conj", "cc", "fixed", "flat", "compound", "list", "parataxis",
                     "orphan", "goeswith", "reparandum", "root", "dep"}

# Enhanced UD: reentrancy (shared arguments) → □ mode edges.
REENTRANT_RELATIONS = {"nsubj:outer", "obj:outer", "nsubj:xsubj"}


@dataclass
class UdToken:
    id:       int
    text:     str
    lemma:    str
    upos:     str          # Universal POS tag
    xpos:     str          # Language-specific POS
    head:     int          # Head token ID (0 = root)
    deprel:   str          # UD dependency relation
    deps:     str = ""     # Enhanced dependencies (for reentrancy)
    feats:    dict = field(default_factory=dict)

    @property
    def is_root(self) -> bool:
        return self.deprel == "root"

    @property
    def is_reentrant(self) -> bool:
        """Shared argument in enhanced UD → □ mode in MTLG."""
        return any(r in self.deps for r in REENTRANT_RELATIONS)


@dataclass
class UdTree:
    tokens:   list[UdToken]
    language: str
    text:     str

    def token_by_id(self, tid: int) -> Optional[UdToken]:
        return next((t for t in self.tokens if t.id == tid), None)

    def dependents_of(self, head_id: int) -> list[UdToken]:
        return [t for t in self.tokens if t.head == head_id]

    def root_tokens(self) -> list[UdToken]:
        return [t for t in self.tokens if t.is_root]


class UdParser:
    """Wraps Stanza for multilingual UD parsing."""

    # Cache of loaded Stanza pipelines (one per language).
    _pipelines: dict = {}

    def __init__(self, language: str = "en"):
        self.language = language
        self._pipeline = self._load_pipeline(language)

    @classmethod
    def _load_pipeline(cls, language: str):
        if language in cls._pipelines:
            return cls._pipelines[language]
        try:
            import stanza
            # Download model if not present.
            stanza.download(language, verbose=False)
            pipeline = stanza.Pipeline(
                language,
                processors="tokenize,mwt,pos,lemma,depparse",
                verbose=False,
                use_gpu=False,
            )
            cls._pipelines[language] = pipeline
            logger.info("Loaded Stanza pipeline for language=%s", language)
            return pipeline
        except Exception as exc:
            logger.error("Failed to load Stanza pipeline for %s: %s", language, exc)
            raise

    def parse(self, sentence: str) -> UdTree:
        """Parse a sentence and return a UdTree."""
        doc = self._pipeline(sentence)
        tokens = []
        for sent in doc.sentences:
            for word in sent.words:
                tokens.append(UdToken(
                    id=word.id,
                    text=word.text,
                    lemma=word.lemma or word.text,
                    upos=word.upos or "X",
                    xpos=word.xpos or "_",
                    head=word.head,
                    deprel=word.deprel or "dep",
                    deps=word.deps or "",
                    feats=dict(f.split("=") for f in (word.feats or "").split("|") if "=" in f),
                ))
        return UdTree(tokens=tokens, language=self.language, text=sentence)

    def parse_batch(self, sentences: list[str]) -> list[UdTree]:
        return [self.parse(s) for s in sentences]


if __name__ == "__main__":
    import json
    parser = UdParser("en")
    tree = parser.parse("Alice runs quickly.")
    for tok in tree.tokens:
        print(json.dumps({
            "id": tok.id, "text": tok.text, "upos": tok.upos,
            "head": tok.head, "deprel": tok.deprel,
        }))


In [ ]:
%%writefile ud_to_mtlg.py
"""Convert UD dependency trees to MTLG modal edge assignments.

Primary dependencies → ◇ (Diamond) mode edges (linear resource use, tree-forming).
Shared arguments (reentrancy in enhanced UD) → □ (Box) mode edges (contraction, DAG).
Long-range / extracted dependencies → ◊ (Lozenge) mode edges (discontinuous).

UCCA categories are assigned based on UD UPOS and deprel:
  - Process/Event: VERB with verbal deprel (root, csubj, ccomp, xcomp, advcl)
  - Participant:   NOUN/PROPN with core dep (nsubj, obj, iobj)
  - State:         ADJ/AUX with predicative function
  - Scene:         clause-level construct (coordinating subgraphs)
  - Adverbial:     advmod, obl
  - Connector:     cc, mark, punct
  - Ground:        discourse, vocative
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Literal

from ud_parser import UdToken, UdTree, UD_CORE_DEPS, UD_NONCORE_DEPS, REENTRANT_RELATIONS

ModalModeStr  = Literal["diamond", "box", "lozenge"]
UccaCategory  = Literal["Scene", "Process", "State", "Participant",
                         "Adverbial", "Connector", "Ground"]


@dataclass
class MtlgEdge:
    src_id:     int
    dst_id:     int
    deprel:     str
    modal_mode: ModalModeStr
    ucca_cat:   UccaCategory
    arity:      int   # remaining args for functor type (0 = saturated)


@dataclass
class MtlgNode:
    token_id:   int
    text:       str
    lemma:      str
    upos:       str
    modal_mode: ModalModeStr
    ucca_cat:   UccaCategory
    arity:      int


@dataclass
class MtlgGraph:
    nodes: list[MtlgNode]
    edges: list[MtlgEdge]
    language: str


def assign_ucca_category(tok: UdToken) -> UccaCategory:
    """Assign UCCA category from UPOS and deprel."""
    upos   = tok.upos.upper()
    deprel = tok.deprel.lower()
    if upos == "VERB" or deprel in ("root", "ccomp", "xcomp", "advcl", "csubj"):
        return "Process"
    if upos in ("NOUN", "PROPN", "PRON") and deprel in UD_CORE_DEPS:
        return "Participant"
    if upos in ("ADJ", "AUX") and deprel in ("amod", "cop", "aux"):
        return "State"
    if deprel in ("advmod", "obl"):
        return "Adverbial"
    if deprel in ("cc", "mark", "punct"):
        return "Connector"
    if deprel in ("discourse", "vocative"):
        return "Ground"
    return "Scene"


def assign_modal_mode(tok: UdToken, deprel: str) -> ModalModeStr:
    """Assign modal mode from deprel and enhanced UD features."""
    if tok.is_reentrant or any(r in deprel for r in REENTRANT_RELATIONS):
        return "box"       # □ — shared argument (contraction)
    if deprel in ("acl:relcl", "nsubj:outer", "obj:outer"):
        return "lozenge"   # ◊ — long-range / extracted
    return "diamond"       # ◇ — primary composition (default)


def compute_functor_arity(tok: UdToken, tree: UdTree) -> int:
    """Estimate the number of remaining arguments for a functor node."""
    if tok.upos == "VERB":
        # Count core dependents (each is one argument slot).
        return len([d for d in tree.dependents_of(tok.id) if d.deprel in UD_CORE_DEPS])
    if tok.upos in ("ADP", "SCONJ"):
        return 1  # prepositions/complementizers take one argument
    return 0  # atomic


def ud_tree_to_mtlg(tree: UdTree) -> MtlgGraph:
    """Convert a UdTree to an MtlgGraph with modal mode and UCCA category assignments."""
    nodes = []
    for tok in tree.tokens:
        ucca_cat   = assign_ucca_category(tok)
        modal_mode = "diamond"  # nodes default to ◇; edges carry the actual mode
        arity      = compute_functor_arity(tok, tree)
        nodes.append(MtlgNode(
            token_id=tok.id, text=tok.text, lemma=tok.lemma,
            upos=tok.upos, modal_mode=modal_mode, ucca_cat=ucca_cat, arity=arity,
        ))

    edges = []
    for tok in tree.tokens:
        if tok.head == 0:
            continue  # skip root
        head_tok = tree.token_by_id(tok.head)
        if head_tok is None:
            continue
        ucca_cat   = assign_ucca_category(tok)
        modal_mode = assign_modal_mode(tok, tok.deprel)
        edges.append(MtlgEdge(
            src_id=tok.head, dst_id=tok.id, deprel=tok.deprel,
            modal_mode=modal_mode, ucca_cat=ucca_cat,
            arity=compute_functor_arity(head_tok, tree),
        ))

    return MtlgGraph(nodes=nodes, edges=edges, language=tree.language)


if __name__ == "__main__":
    import json
    from ud_parser import UdParser

    parser = UdParser("en")
    tree   = parser.parse("Alice runs quickly.")
    mtlg   = ud_tree_to_mtlg(tree)
    for edge in mtlg.edges:
        print(json.dumps({
            "src": edge.src_id, "dst": edge.dst_id,
            "deprel": edge.deprel, "mode": edge.modal_mode, "ucca": edge.ucca_cat,
        }))


In [ ]:
%%writefile mtlg_inducer.py
"""Probabilistic MTLG grammar induction.

Extends Kwiatkowski et al. (2010) higher-order unification to multimodal types.
Bisk & Hockenmaier HDP-CCG extended to modal categories (◇, □, ◊).

Pipeline:
  UD tree → MTLG graph → extract (word, modal_type) pairs → update lexicon counts → normalize.

Output: per-language modal lexicon stored as JSON.
  {"lemma": {"modal_mode": "diamond", "ucca_cat": "Process", "arity": 2, "count": 145}, ...}
"""
from __future__ import annotations

import json
import logging
import math
from collections import defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterator

from ud_to_mtlg import MtlgGraph, MtlgNode, ud_tree_to_mtlg
from morphological_fst import preprocess_for_type_assignment

logger = logging.getLogger(__name__)


@dataclass
class LexTypeEntry:
    modal_mode: str
    ucca_cat:   str
    arity:      int
    count:      int = 0
    log_prob:   float = 0.0


@dataclass
class PerLanguageLexicon:
    language: str
    entries:  dict[str, list[LexTypeEntry]] = field(default_factory=lambda: defaultdict(list))

    def update(self, lemma: str, mode: str, cat: str, arity: int, weight: float = 1.0):
        for entry in self.entries[lemma]:
            if entry.modal_mode == mode and entry.ucca_cat == cat and entry.arity == arity:
                entry.count += weight
                return
        self.entries[lemma].append(LexTypeEntry(mode, cat, arity, count=weight))

    def normalize(self):
        """Compute log-probabilities from counts (MLE)."""
        for lemma, type_entries in self.entries.items():
            total = sum(e.count for e in type_entries)
            if total == 0:
                continue
            for e in type_entries:
                e.log_prob = math.log(e.count / total)

    def best_type(self, lemma: str) -> LexTypeEntry | None:
        entries = self.entries.get(lemma)
        if not entries:
            return None
        return max(entries, key=lambda e: e.count)

    def to_dict(self) -> dict:
        return {
            lemma: [
                {"modal_mode": e.modal_mode, "ucca_cat": e.ucca_cat,
                 "arity": e.arity, "count": e.count, "log_prob": e.log_prob}
                for e in entries
            ]
            for lemma, entries in self.entries.items()
        }

    def save(self, path: Path):
        path.write_text(json.dumps(self.to_dict(), ensure_ascii=False, indent=2))
        logger.info("Saved lexicon for %s to %s (%d lemmas)", self.language, path, len(self.entries))

    @classmethod
    def load(cls, language: str, path: Path) -> "PerLanguageLexicon":
        data = json.loads(path.read_text())
        lex = cls(language=language)
        for lemma, entries in data.items():
            for e in entries:
                lex.entries[lemma].append(LexTypeEntry(
                    e["modal_mode"], e["ucca_cat"], e["arity"],
                    count=e["count"], log_prob=e["log_prob"],
                ))
        return lex


class MtlgInducer:
    """Induces a probabilistic MTLG lexicon from MTLG graphs derived from UD trees."""

    def __init__(self, language: str):
        self.language = language
        self.lexicon  = PerLanguageLexicon(language=language)
        self._trees_processed = 0

    def observe_graph(self, graph: MtlgGraph):
        """Update lexicon counts from one MTLG graph."""
        for node in graph.nodes:
            # Weight by: arity > 0 gives more signal (functors)
            weight = 1.5 if node.arity > 0 else 1.0
            self.lexicon.update(node.lemma, node.modal_mode, node.ucca_cat, node.arity, weight)
        self._trees_processed += 1

    def induce_from_stream(
        self,
        trees: Iterator[MtlgGraph],
        max_trees: int = 10_000,
    ) -> PerLanguageLexicon:
        """Induce lexicon from a stream of MTLG graphs."""
        for i, graph in enumerate(trees):
            if i >= max_trees:
                break
            self.observe_graph(graph)
            if i % 1_000 == 0:
                logger.info("Processed %d trees for %s", i, self.language)
        self.lexicon.normalize()
        logger.info("Induced %d lemmas for %s from %d trees",
                    len(self.lexicon.entries), self.language, self._trees_processed)
        return self.lexicon

    def parse_accuracy(self, test_graphs: list[MtlgGraph]) -> float:
        """Estimate parse accuracy: fraction of nodes where best-predicted type matches gold."""
        if not test_graphs:
            return 0.0
        correct = total = 0
        for graph in test_graphs:
            for node in graph.nodes:
                total += 1
                best = self.lexicon.best_type(node.lemma)
                if best and best.modal_mode == node.modal_mode and best.ucca_cat == node.ucca_cat:
                    correct += 1
        return correct / total if total > 0 else 0.0


def run_induction_pipeline(
    language:    str,
    max_samples: int = 5_000,
    output_dir:  str = "lexicons",
) -> PerLanguageLexicon:
    """End-to-end induction: mC4 stream → UD parse → MTLG graph → lexicon."""
    from mc4_stream import stream_mc4, requires_fst
    from ud_parser import UdParser

    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    parser  = UdParser(language)
    inducer = MtlgInducer(language)

    def graph_stream():
        for item in stream_mc4(language, max_samples=max_samples):
            tokens = preprocess_for_type_assignment(item["text"], language)
            sentence = " ".join(t.split("[")[0] for t in tokens)  # strip FST tags for parser
            try:
                tree  = parser.parse(sentence)
                graph = ud_tree_to_mtlg(tree)
                yield graph
            except Exception as exc:
                logger.debug("Parse error: %s", exc)

    lex = inducer.induce_from_stream(graph_stream(), max_trees=max_samples)
    lex.save(out_path / f"{language}_lexicon.json")
    return lex


if __name__ == "__main__":
    import sys
    lang = sys.argv[1] if len(sys.argv) > 1 else "en"
    lex  = run_induction_pipeline(lang, max_samples=100)
    print(f"Induced {len(lex.entries)} lemmas for {lang}")
    # Show top 5 by count
    top = sorted(lex.entries.items(), key=lambda kv: sum(e.count for e in kv[1]), reverse=True)[:5]
    for lemma, entries in top:
        best = max(entries, key=lambda e: e.count)
        print(f"  {lemma}: mode={best.modal_mode} cat={best.ucca_cat} arity={best.arity} count={best.count:.0f}")


In [ ]:
%%writefile trd_bootstrap.py
"""TRD bootstrapping: crystallize co-occurring UCCA-category type profiles into TRD entries.

A TRD (Transient Relative Domain) is a cluster of situation types, infon patterns,
and MTLG modal type profiles that co-activate together.

Algorithm:
  1. Process dissolved TRs from the LCS pipeline (per language).
  2. Collect (modal_mode, ucca_cat) co-occurrence vectors per sentence/context.
  3. Cluster co-occurrence vectors (k-means over the modal profile space).
  4. Each cluster → one TRD entry: {type_profile, modal_operator_inventory}.
  5. Validate: > 70% of held-out situations map to a known TRD.

TRD vocabulary target: < 1000 entries per language family (to prevent fragmentation).
"""
from __future__ import annotations

import json
import logging
import math
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np

from ud_to_mtlg import MtlgGraph

logger = logging.getLogger(__name__)

MAX_TRD_VOCAB = 1000
N_CLUSTERS_DEFAULT = 64  # starting point; tuned via held-out coverage


@dataclass
class TrdEntry:
    trd_id:      int
    label:       str
    modal_profile: dict[str, float]   # (mode, cat) key → frequency
    infon_patterns: list[str]
    support:     int = 0

    def to_dict(self) -> dict:
        return {
            "trd_id": self.trd_id,
            "label":  self.label,
            "modal_profile": self.modal_profile,
            "infon_patterns": self.infon_patterns,
            "support": self.support,
        }


def graph_to_modal_vector(graph: MtlgGraph, vocab: list[str]) -> np.ndarray:
    """Convert a graph's modal type distribution to a fixed-size vector."""
    counts: Counter = Counter()
    for edge in graph.edges:
        key = f"{edge.modal_mode}_{edge.ucca_cat}"
        counts[key] += 1
    total = sum(counts.values()) or 1
    return np.array([counts.get(v, 0) / total for v in vocab], dtype=np.float32)


def build_vocabulary(graphs: list[MtlgGraph]) -> list[str]:
    """Build the modal-profile vocabulary from observed (mode, cat) pairs."""
    keys: set = set()
    for g in graphs:
        for e in g.edges:
            keys.add(f"{e.modal_mode}_{e.ucca_cat}")
    return sorted(keys)


class KMeansTrd:
    """Simple k-means clustering for TRD crystallization."""

    def __init__(self, n_clusters: int, max_iter: int = 20, seed: int = 42):
        self.n_clusters = n_clusters
        self.max_iter   = max_iter
        self.rng        = np.random.default_rng(seed)
        self.centroids: np.ndarray | None = None

    def fit(self, vectors: np.ndarray) -> np.ndarray:
        """Returns cluster assignments for each vector."""
        n = len(vectors)
        k = min(self.n_clusters, n)
        # Initialize centroids with k-means++ heuristic.
        idx = [self.rng.integers(0, n)]
        for _ in range(k - 1):
            dists = np.array([min(np.linalg.norm(v - vectors[i]) ** 2 for i in idx) for v in vectors])
            probs = dists / dists.sum()
            idx.append(self.rng.choice(n, p=probs))
        self.centroids = vectors[idx]

        assignments = np.zeros(n, dtype=int)
        for _ in range(self.max_iter):
            # Assign each vector to nearest centroid.
            dists = np.linalg.norm(vectors[:, None] - self.centroids[None, :], axis=2)
            new_assignments = dists.argmin(axis=1)
            if np.array_equal(new_assignments, assignments):
                break
            assignments = new_assignments
            # Update centroids.
            for c in range(k):
                members = vectors[assignments == c]
                if len(members) > 0:
                    self.centroids[c] = members.mean(axis=0)
        return assignments

    def predict(self, vector: np.ndarray) -> int:
        if self.centroids is None:
            raise RuntimeError("Fit the model first.")
        dists = np.linalg.norm(self.centroids - vector, axis=1)
        return int(dists.argmin())


class TrdBootstrapper:
    def __init__(self, n_clusters: int = N_CLUSTERS_DEFAULT):
        self.n_clusters = n_clusters
        self.vocab:    list[str] = []
        self.trds:     list[TrdEntry] = []
        self._model:   KMeansTrd | None = None

    def bootstrap(self, graphs: list[MtlgGraph], language: str) -> list[TrdEntry]:
        """Crystallize TRDs from a batch of MTLG graphs."""
        if not graphs:
            return []
        self.vocab = build_vocabulary(graphs)
        vectors    = np.stack([graph_to_modal_vector(g, self.vocab) for g in graphs])
        n_clusters = min(self.n_clusters, len(graphs), MAX_TRD_VOCAB)
        self._model = KMeansTrd(n_clusters)
        assignments = self._model.fit(vectors)
        self.trds = self._build_trd_entries(graphs, assignments, n_clusters, language)
        logger.info("Bootstrapped %d TRDs for %s from %d graphs", len(self.trds), language, len(graphs))
        return self.trds

    def _build_trd_entries(
        self, graphs: list[MtlgGraph], assignments: np.ndarray,
        n_clusters: int, language: str,
    ) -> list[TrdEntry]:
        cluster_graphs: dict[int, list[MtlgGraph]] = defaultdict(list)
        for i, g in enumerate(graphs):
            cluster_graphs[int(assignments[i])].append(g)

        trds = []
        for cluster_id, cg in cluster_graphs.items():
            # Build modal profile from cluster centroid.
            centroid = self._model.centroids[cluster_id]
            profile  = {v: float(centroid[i]) for i, v in enumerate(self.vocab) if centroid[i] > 0.01}
            # Characteristic infon patterns: most common lemmas in this cluster.
            lemma_counter: Counter = Counter()
            for g in cg:
                for node in g.nodes:
                    if node.ucca_cat in ("Process", "Participant"):
                        lemma_counter[node.lemma] += 1
            top_lemmas = [l for l, _ in lemma_counter.most_common(5)]
            trds.append(TrdEntry(
                trd_id=cluster_id,
                label=f"{language}_trd_{cluster_id}",
                modal_profile=profile,
                infon_patterns=top_lemmas,
                support=len(cg),
            ))
        return trds

    def coverage(self, held_out: list[MtlgGraph]) -> float:
        """Fraction of held-out graphs that map to a known TRD."""
        if not held_out or self._model is None:
            return 0.0
        matched = 0
        for g in held_out:
            vec = graph_to_modal_vector(g, self.vocab)
            trd_id = self._model.predict(vec)
            if any(t.trd_id == trd_id for t in self.trds):
                matched += 1
        return matched / len(held_out)

    def save(self, path: Path):
        data = {
            "vocab": self.vocab,
            "trds":  [t.to_dict() for t in self.trds],
        }
        path.write_text(json.dumps(data, ensure_ascii=False, indent=2))
        logger.info("Saved %d TRDs to %s", len(self.trds), path)


if __name__ == "__main__":
    import sys
    from ud_to_mtlg import ud_tree_to_mtlg
    from ud_parser import UdParser
    from mc4_stream import stream_mc4
    from morphological_fst import preprocess_for_type_assignment

    lang = sys.argv[1] if len(sys.argv) > 1 else "en"
    parser = UdParser(lang)
    graphs = []
    for item in stream_mc4(lang, max_samples=200):
        try:
            tokens   = preprocess_for_type_assignment(item["text"], lang)
            sentence = " ".join(t.split("[")[0] for t in tokens)
            tree     = parser.parse(sentence)
            graphs.append(ud_tree_to_mtlg(tree))
        except Exception:
            pass

    train, held_out = graphs[:150], graphs[150:]
    bootstrapper = TrdBootstrapper(n_clusters=16)
    trds = bootstrapper.bootstrap(train, lang)
    cov  = bootstrapper.coverage(held_out)
    print(f"TRDs: {len(trds)}, coverage on held-out: {cov:.2%}")
    bootstrapper.save(Path(f"{lang}_trds.json"))


## 3 · Demo Configuration

In [ ]:
# ── Tune these for a faster / richer demo ───────────────────────────────────
LANGUAGE       = "en"   # ISO 639-1  (any mC4 language)
TRAIN_SAMPLES  = 500    # sentences used for lexicon + TRD induction
HELD_OUT       = 50     # sentences reserved for evaluation
N_TRD_CLUSTERS = 16     # TRD k-means clusters  (16 ≈ fast demo; try 64 for full run)
LEXICON_PATH   = f"{LANGUAGE}_lexicon.json"
TRD_PATH       = f"{LANGUAGE}_trds.json"

print(f"Config · lang={LANGUAGE}  train={TRAIN_SAMPLES}  held-out={HELD_OUT}  k={N_TRD_CLUSTERS}")


## 4 · Download Stanza Language Model

In [ ]:
import stanza, logging
logging.basicConfig(level=logging.WARNING)   # suppress verbose output
stanza.download(LANGUAGE, verbose=False)
print(f"Stanza '{LANGUAGE}' model ready.")


## 5 · Stream C4 Data

We use HuggingFace `datasets` in **streaming mode** — only the sentences we need are downloaded, not the entire corpus.


In [ ]:
import sys, time
sys.path.insert(0, ".")          # make local modules importable

from mc4_stream import stream_mc4

N_TOTAL = TRAIN_SAMPLES + HELD_OUT
print(f"Streaming {N_TOTAL} sentences from allenai/c4 ({LANGUAGE}) …")

t0 = time.time()
raw_sentences = []
for item in stream_mc4(LANGUAGE, max_samples=N_TOTAL):
    raw_sentences.append(item["text"])
    if len(raw_sentences) % 100 == 0:
        print(f"  {len(raw_sentences)} / {N_TOTAL} …", flush=True)

elapsed = time.time() - t0
print(f"\nStreamed {len(raw_sentences)} sentences in {elapsed:.1f}s")
print("\nSample sentences:")
for s in raw_sentences[:3]:
    print(f"  • {s[:110]}")


## 6 · Parse & Build MTLG Graphs

Each sentence goes through:
- **Morphological preprocessing** (FST for agglutinative languages; passthrough for English)
- **Stanza UD parser** → dependency tree
- **UD → MTLG converter** → modal graph with UCCA category labels


In [ ]:
from ud_parser import UdParser
from ud_to_mtlg import ud_tree_to_mtlg
from morphological_fst import preprocess_for_type_assignment

parser = UdParser(LANGUAGE)
print("Parser loaded. Building MTLG graphs …")

graphs, errors = [], 0
t0 = time.time()

for i, text in enumerate(raw_sentences):
    try:
        tokens   = preprocess_for_type_assignment(text, LANGUAGE)
        sentence = " ".join(t.split("[")[0] for t in tokens)
        tree     = parser.parse(sentence)
        graph    = ud_tree_to_mtlg(tree)
        if graph.nodes:
            graphs.append(graph)
    except Exception:
        errors += 1
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(raw_sentences)} → {len(graphs)} graphs  ({errors} errors)")

print(f"\nDone: {len(graphs)} valid graphs in {time.time()-t0:.1f}s  ({errors} parse errors)")

train_graphs    = graphs[:TRAIN_SAMPLES]
held_out_graphs = graphs[TRAIN_SAMPLES : TRAIN_SAMPLES + HELD_OUT]
print(f"Split → train: {len(train_graphs)}  |  held-out: {len(held_out_graphs)}")


## 7 · Induce MTLG Lexicon

The `MtlgInducer` observes each MTLG graph and accumulates `(lemma, modal_mode, ucca_category, arity) → count` statistics, then normalises to log-probabilities (MLE).


In [ ]:
from pathlib import Path
from mtlg_inducer import MtlgInducer

print("Inducing MTLG lexicon …")
t0 = time.time()

inducer = MtlgInducer(LANGUAGE)
lexicon = inducer.induce_from_stream(iter(train_graphs), max_trees=TRAIN_SAMPLES)
lexicon.save(Path(LEXICON_PATH))

print(f"Lexicon: {len(lexicon.entries)} lemmas  |  {time.time()-t0:.1f}s")
print(f"Saved  → {LEXICON_PATH}\n")

# Top-10 by observation count
top10 = sorted(lexicon.entries.items(),
               key=lambda kv: sum(e.count for e in kv[1]), reverse=True)[:10]

MODE_SYM = {"diamond": "◇", "box": "□", "lozenge": "◊"}
print(f"{'Lemma':<16} {'Mode':<11} {'UCCA-Cat':<14} {'Arity'} {'Count':>7}  LogProb")
print("─" * 65)
for lemma, entries in top10:
    best = max(entries, key=lambda e: e.count)
    sym  = MODE_SYM.get(best.modal_mode, "?")
    print(f"{lemma:<16} {sym} {best.modal_mode:<9} {best.ucca_cat:<14} {best.arity}  "
          f"{best.count:>7.0f}  {best.log_prob:.3f}")


## 8 · Bootstrap TRD Clusters

Each MTLG graph is projected to a **modal type profile vector** (normalised (mode, ucca_cat) co-occurrence counts). K-means clustering groups these vectors into **TRDs** — situation-type domains that the reasoning engine uses to adapt its activation thresholds.


In [ ]:
from trd_bootstrap import TrdBootstrapper

print(f"Bootstrapping {N_TRD_CLUSTERS} TRD clusters …")
t0 = time.time()

bootstrapper = TrdBootstrapper(n_clusters=N_TRD_CLUSTERS)
trds         = bootstrapper.bootstrap(train_graphs, LANGUAGE)
coverage     = bootstrapper.coverage(held_out_graphs)
bootstrapper.save(Path(TRD_PATH))

print(f"TRDs: {len(trds)} clusters  |  {time.time()-t0:.1f}s")
print(f"Held-out TRD coverage: {coverage:.2%}")
print(f"Saved → {TRD_PATH}\n")

print(f"{'TRD label':<24} {'Support':>7}  Top infon patterns")
print("─" * 60)
for trd in sorted(trds, key=lambda t: t.support, reverse=True)[:8]:
    pats = ", ".join(trd.infon_patterns[:4]) or "(none)"
    print(f"{trd.label:<24} {trd.support:>7}  {pats}")


## 9 · Evaluation

- **Parse accuracy** — fraction of held-out nodes where the lexicon's   best-predicted `(modal_mode, ucca_cat)` matches the gold assignment from the UD parser.
- **TRD coverage** — fraction of held-out graphs that map to a known TRD cluster.


In [ ]:
from collections import Counter

parse_acc = inducer.parse_accuracy(held_out_graphs)

print(f"Parse accuracy  (held-out): {parse_acc:.2%}")
print(f"TRD coverage    (held-out): {coverage:.2%}")

# Modal mode + UCCA distribution over training set
mode_counts, ucca_counts = Counter(), Counter()
for g in train_graphs:
    for e in g.edges:
        mode_counts[e.modal_mode] += 1
        ucca_counts[e.ucca_cat]   += 1

MODE_SYM = {"diamond": "◇", "box": "□", "lozenge": "◊"}

def bar(frac, width=30):
    return "█" * int(width * frac)

total_e = sum(mode_counts.values()) or 1
total_u = sum(ucca_counts.values()) or 1

print("\nModal mode distribution (training edges):")
for mode, cnt in sorted(mode_counts.items(), key=lambda x: -x[1]):
    sym = MODE_SYM.get(mode, "?")
    print(f"  {sym} {mode:<10} {cnt:>6}  {bar(cnt/total_e)}")

print("\nUCCA category distribution (training edges):")
for cat, cnt in sorted(ucca_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat:<14} {cnt:>6}  {bar(cnt/total_u)}")


## 10 · Distribution Visualisation

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Modal mode pie chart
ax = axes[0]
labels = list(mode_counts.keys())
sizes  = [mode_counts[l] for l in labels]
symbols = [MODE_SYM.get(l, "?") + " " + l for l in labels]
colors = ["#4C72B0", "#DD8452", "#55A868"][:len(labels)]
ax.pie(sizes, labels=symbols, colors=colors, autopct="%1.1f%%",
       startangle=140, textprops={"fontsize": 10})
ax.set_title("Modal Mode Distribution\n(Training Edges)", fontweight="bold")

# UCCA category horizontal bar
ax = axes[1]
cats = list(ucca_counts.keys())
cnts = [ucca_counts[c] for c in cats]
sorted_pairs = sorted(zip(cnts, cats), reverse=True)
cnts_s, cats_s = zip(*sorted_pairs)
bars = ax.barh(cats_s, cnts_s, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Edge count")
ax.set_title("UCCA Category Distribution\n(Training Edges)", fontweight="bold")
ax.bar_label(bars, fmt="%d", padding=4, fontsize=8)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig("modal_distribution.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved → modal_distribution.png")


## 11 · Inference Demo

The trained model can now **analyse any new English sentence**:

1. Preprocess & UD-parse
2. Map lemmas through the induced lexicon → best `(mode, ucca_cat)`
3. Assign the whole sentence to a TRD cluster
4. Display the structured modal-semantic breakdown


In [ ]:
import numpy as np
from trd_bootstrap import graph_to_modal_vector

MODE_SYM = {"diamond": "◇", "box": "□", "lozenge": "◊"}

def analyze(sentence: str, lang: str = LANGUAGE) -> dict:
    """Full RAG inference: sentence → MTLG modal analysis + TRD assignment."""
    tokens   = preprocess_for_type_assignment(sentence, lang)
    clean    = " ".join(t.split("[")[0] for t in tokens)
    tree     = parser.parse(clean)
    graph    = ud_tree_to_mtlg(tree)

    # Lexicon lookup per node
    node_rows = []
    for node in graph.nodes:
        entry = lexicon.best_type(node.lemma)
        node_rows.append({
            "text":      node.text,
            "lemma":     node.lemma,
            "upos":      node.upos,
            "ucca_cat":  node.ucca_cat,
            "arity":     node.arity,
            "lex_mode":  entry.modal_mode if entry else "—",
            "lex_cat":   entry.ucca_cat   if entry else "—",
            "lex_count": entry.count      if entry else 0,
        })

    # TRD assignment
    trd_label, trd_patterns = "unknown", []
    if bootstrapper._model and bootstrapper.vocab:
        vec       = graph_to_modal_vector(graph, bootstrapper.vocab)
        trd_id    = bootstrapper._model.predict(vec)
        for t in bootstrapper.trds:
            if t.trd_id == trd_id:
                trd_label    = t.label
                trd_patterns = t.infon_patterns

    edge_rows = [{"deprel": e.deprel, "mode": e.modal_mode, "ucca": e.ucca_cat}
                 for e in graph.edges]

    return {"sentence": sentence, "trd": trd_label,
            "trd_patterns": trd_patterns, "nodes": node_rows, "edges": edge_rows}


def print_analysis(result: dict):
    print(f"\n{'═'*70}")
    print(f"  Sentence  : {result['sentence']}")
    print(f"  TRD       : {result['trd']}")
    if result["trd_patterns"]:
        print(f"  TRD keys  : {', '.join(result['trd_patterns'][:5])}")
    print(f"  {'Token':<16} {'UPOS':<7} {'UCCA':<13} {'Sym'} {'Lex-mode':<10} {'Arity'} {'#Obs'}")
    print(f"  {'─'*64}")
    for n in result["nodes"]:
        sym = MODE_SYM.get(n["lex_mode"], " ")
        print(f"  {n['text']:<16} {n['upos']:<7} {n['ucca_cat']:<13} {sym}  "
              f"{n['lex_mode']:<10} {n['arity']}     {n['lex_count']:.0f}")
    print(f"\n  Edges ({len(result['edges'])}):")
    for e in result["edges"][:6]:
        sym = MODE_SYM.get(e["mode"], "?")
        print(f"    [{e['deprel']}]  {sym} {e['mode']}  →  {e['ucca']}")
    print(f"{'═'*70}")


print("Inference pipeline ready.")


In [ ]:
demo_sentences = [
    "The scientist discovered a new particle in the laboratory.",
    "Climate change threatens biodiversity across the planet.",
    "Alice quickly ran to the store and bought fresh bread.",
    "The government passed new legislation on digital privacy.",
    "Deep learning models require large amounts of training data.",
    "She smiled and waved goodbye to her friends.",
]

print("INFERENCE DEMO — RAG MTLG Semantic Analysis")
for sentence in demo_sentences:
    result = analyze(sentence)
    print_analysis(result)


## 12 · MTLG Graph Visualisation

NetworkX renders the modal dependency graph for the first demo sentence. Node colours encode UCCA categories; edge styles encode modal modes (solid = ◇ diamond, dashed = □ box, dotted = ◊ lozenge).


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

TARGET = demo_sentences[0]

tokens   = preprocess_for_type_assignment(TARGET, LANGUAGE)
clean    = " ".join(t.split("[")[0] for t in tokens)
tree     = parser.parse(clean)
g        = ud_tree_to_mtlg(tree)

G = nx.DiGraph()
for node in g.nodes:
    G.add_node(node.token_id, label=node.text, ucca=node.ucca_cat)
for edge in g.edges:
    G.add_edge(edge.src_id, edge.dst_id, mode=edge.modal_mode, deprel=edge.deprel)

UCCA_COLORS = {
    "Process": "#DD8452", "Participant": "#4C72B0", "Scene": "#55A868",
    "State": "#C44E52",   "Adverbial":  "#8172B3",
    "Connector": "#937860", "Ground": "#DA8BC3",
}
MODE_STYLE = {"diamond": "solid", "box": "dashed",   "lozenge": "dotted"}
MODE_COLOR = {"diamond": "#4C72B0","box": "#DD8452","lozenge": "#55A868"}

node_colors = [UCCA_COLORS.get(G.nodes[n].get("ucca", "Scene"), "#aaaaaa") for n in G.nodes]
pos = nx.spring_layout(G, seed=42, k=2.2)

fig, ax = plt.subplots(figsize=(14, 6))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1000, ax=ax, alpha=0.95)
nx.draw_networkx_labels(G, pos,
    labels={n: G.nodes[n].get("label", str(n)) for n in G.nodes},
    font_size=8, font_color="white", font_weight="bold", ax=ax)

for (u, v, data) in G.edges(data=True):
    mode = data.get("mode", "diamond")
    nx.draw_networkx_edges(G, pos, edgelist=[(u, v)],
        style=MODE_STYLE.get(mode, "solid"),
        edge_color=MODE_COLOR.get(mode, "#888"),
        arrows=True, arrowsize=20, width=2, ax=ax,
        connectionstyle="arc3,rad=0.1")

edge_labels = {(e.src_id, e.dst_id): e.deprel for e in g.edges}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7, ax=ax)

legend_patches = [mpatches.Patch(color=c, label=k) for k, c in UCCA_COLORS.items()]
mode_lines = [
    plt.Line2D([0],[0], color=MODE_COLOR[m], lw=2,
               linestyle=MODE_STYLE[m], label=f"{MODE_SYM[m]} {m}")
    for m in ("diamond","box","lozenge")
]
ax.legend(handles=legend_patches + mode_lines, loc="lower left", fontsize=8,
          title="UCCA / Mode", ncol=2)
ax.set_title(f'MTLG Graph: "{TARGET}"', fontsize=11, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig("mtlg_graph.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved → mtlg_graph.png")


## 13 · Download Trained Artefacts

In [ ]:
# Download lexicon and TRD files from Colab to your machine
from google.colab import files
files.download(LEXICON_PATH)
files.download(TRD_PATH)
print("Download triggered for:", LEXICON_PATH, TRD_PATH)
